In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [2]:
# b2p_matched_data_file = Path(project_root / "data/people/prepping4loading/b2p_matched_data.json")
b2p_remapped_data_file = Path(project_root / "data/people/prepping4loading/b2p_remapped_data.json")

b2p_new_data_file = Path(project_root / "data/people/fixing_ids/b2p_data.json")

# b2p_matched_list_file = Path(project_root / "data/people/prepping4loading/b2p_matched_list.json")
b2p_remapped_list_file = Path(project_root / "data/people/prepping4loading/b2p_remapped_list.json")

rprint(b2p_remapped_list_file)

b2p_new_list_file = Path(project_root / "data/people/fixing_ids/b2p_list.json")

with open(b2p_remapped_list_file, "r") as f:
   b2p_m = json.load(f)

with open(b2p_new_list_file, "r") as f:
   b2p_n = json.load(f)

b2p_list = b2p_m + b2p_n

rprint(f"""
=== info ===
b2p_m length: {len(b2p_m)}
b2p_n length: {len(b2p_n)}
b2p_combined len: {len(b2p_list)}
math check: {len(b2p_m)+len(b2p_n)}
=== DONE ===
""")
b2p_list_file = Path(project_root / "data/people/prepping4loading/b2p_list_complete.json")

with open(b2p_list_file, "w") as f:
    json.dump(b2p_list, f, ensure_ascii=False, indent=2)


/Users/mariavalencia/Documents/curiosity corner/I love 
learning/bibliopa/data/people/prepping4loading/b2p_remapped_list.json

=== info ===
b2p_m length: 9234
b2p_n length: 2390
b2p_combined len: 11624
math check: 11624
=== DONE ===

In [3]:
books_db_file = Path(project_root / "data/from db/books_db.json")
with open(books_db_file, "r") as f:
   books_db = json.load(f)

with open(b2p_list_file, "r") as f:
   b2p_list = json.load(f)



# rprint(books_db[:3])
rprint(f"len books_db: {len(books_db)}")


# book_id_lookup = {b["composite_id"]: b for k, b in books_db}
book_id_dict = {v["composite_id"]: v["book_id"] for v in books_db}

rprint(f"len book_id_dict: {len(book_id_dict)}")

# rprint(dict(list(book_id_dict.items())[:3]))

b2p_with_books = []
seen_combinations = set()
update_count = 0
problems = {}
roles = [
    "is_author",
    "is_editor",
    "is_contributor",
    "is_translator"
]
updates = {}
problem_count = 0

for entry in b2p_list:
# for entry in b2p_list[:20]:
    composite_id = entry["composite_id"]
    person_id = entry["person_id"]
    unified_id = entry["unified_id"]
    display_name = entry["display_name"]
    sort_order = entry["sort_order"]
    is_author = entry["is_author"]
    is_editor = entry["is_editor"]
    is_contributor = entry["is_contributor"]
    is_translator = entry["is_translator"]

    if not composite_id in book_id_dict:
        problem_count += 1
        problems[composite_id] = entry
        continue

    book_id = book_id_dict[composite_id]


    combo = (person_id, composite_id)
    if combo not in seen_combinations:
        seen_combinations.add(combo)
        b2p_with_books.append({
            "book_id": book_id,
            "composite_id": composite_id,
            "person_id": person_id,
            "unified_id": unified_id,
            "display_name": display_name,
            "sort_order": sort_order,
            "is_author": is_author,
            "is_editor": is_editor,
            "is_contributor": is_contributor,
            "is_translator": is_translator,
        })
    else:
        existing = [e for e in b2p_with_books if e["person_id"] == person_id and e["composite_id"] == composite_id][0]

        existing.update({
            role: existing[role] or entry[role] for role in roles
        })
        update_count += 1

b2p_loading_file = Path(project_root / "data_reload/b2p_loading_file.json")

with open(b2p_loading_file, "w") as f:
    json.dump(b2p_with_books, f, ensure_ascii=False, indent=2)


rprint(f"""
=== INFO ===
problem count: {problem_count}
update_count: {update_count}
total before: {len(b2p_list)}
total entries with books: {len(b2p_with_books)}
=== DONE ===
""")
rprint(dict(list(problems.items())[:5]))


len books_db: 10927

len book_id_dict: 10927

=== INFO ===
problem count: 0
update_count: 0
total before: 11624
total entries with books: 11624
=== DONE ===

{}